# Exploring the MATH dataset

### Load the dataset

In [1]:
from datasets import load_dataset, concatenate_datasets

subjects = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

all_train = []
all_test = []

for subject in subjects:
    ds = load_dataset("EleutherAI/hendrycks_math", subject)
    all_train.append(ds["train"])
    all_test.append(ds["test"])

train_ds = concatenate_datasets(all_train)
test_ds = concatenate_datasets(all_test)

print(train_ds)
print(test_ds)
example = train_ds[0]
print(example)

Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 7500
})
Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 5000
})
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.'}


### Pretty Print an Example

In [2]:
from IPython.display import display, Markdown
from math_verify import parse, verify


def view_problem(example, show_solution=True):
    gold = parse(example["solution"])[0]
    md = f"""
## {example["type"]} — {example["level"]}

### Problem

{example["problem"]}
"""
    if show_solution:
        md += f"""

### Solution

{example["solution"]}

### Parsed Solution

{gold}

"""
    display(Markdown(md))


view_problem(example)


## Algebra — Level 5

### Problem

Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).


### Solution

For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\boxed{0}$.

### Parsed Solution

0



### Verify Model's Output

In [3]:
def verify_math_llm_answer(problem: dict, llm_answer: str) -> bool:
    if "solution" not in problem:
        raise KeyError("problem must contain a 'solution' field")

    gold = parse(problem["solution"])
    pred = parse(llm_answer)

    return verify(gold, pred)

model_output = r"We solve the equation and get \boxed{0}."
verify_math_llm_answer(example, model_output)

True

### Model Calling

In [4]:
import os
print({k: bool(os.getenv(k)) for k in
       ["AWS_BEARER_TOKEN_BEDROCK", "AWS_REGION", "CLAUDE_CODE_USE_BEDROCK"]})

{'AWS_BEARER_TOKEN_BEDROCK': True, 'AWS_REGION': True, 'CLAUDE_CODE_USE_BEDROCK': True}


In [5]:
from anthropic import AnthropicBedrock

# AnthropicBedrock auto-reads AWS_REGION and AWS_BEARER_TOKEN_BEDROCK from env.
# It also accepts standard AWS profile/key auth if you ever switch.
client = AnthropicBedrock()

# Bedrock model IDs differ from the Anthropic-API short names.
# Haiku 4.5 on Bedrock:
HAIKU_BEDROCK_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

resp = client.messages.create(
    model=HAIKU_BEDROCK_ID,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is 17 * 23? Put the final answer in \\boxed{}."}
    ],
)

# resp.content is a list of content blocks
print(resp.content[0].text)
print("---")
print("input tokens:", resp.usage.input_tokens, "output tokens:", resp.usage.output_tokens)


I need to calculate 17 × 23.

I'll use the standard multiplication method:

17 × 23 = 17 × (20 + 3)
       = 17 × 20 + 17 × 3
       = 340 + 51
       = 391

\boxed{391}
---
input tokens: 26 output tokens: 80


In [14]:
from claude_agent_sdk import ClaudeAgentOptions, query

HAIKU = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

leaf_options = ClaudeAgentOptions(
    model=HAIKU,

    # Don't read ~/.claude/settings.json, .claude/settings.json, or any other source.
    # This is the single most important line.
    setting_sources=[],

    # Don't connect any MCP server (no Notion, nothing).
    mcp_servers={},

    # Don't expose any tool to the agent.
    allowed_tools=[],

    # Your own system prompt; nothing inherited.
    system_prompt=(
        "You are a math problem solver. Solve the given problem step by step. "
        "Put your final answer inside \\boxed{...}."
    ),

    # Optional: tighten further.
    permission_mode="default",
)


prompt = r"What is $17 \times 23$? Put the final answer in \boxed{}."

async for msg in query(prompt=prompt, options=leaf_options):
    # msg is a structured event: assistant text, tool use, result, etc.
    print(msg)


SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/home/pedro/Documents/workplace/llm_organizations/notebooks', 'session_id': '616ab0cd-4ca8-44ca-b888-c9b6348db67a', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'NotebookEdit', 'Read', 'ScheduleWakeup', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskOutput', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'WebFetch', 'Write'], 'mcp_servers': [], 'model': 'us.anthropic.claude-haiku-4-5-20251001-v1:0', 'permissionMode': 'default', 'slash_commands': ['update-config', 'verify', 'debug', 'code-review', 'batch', 'fewer-permission-prompts', 'loop', 'claude-api', 'run', 'run-skill-generator', 'clear', 'compact', 'context', 'heapdump', 'init', 'review', 'security-review', 'usage', 'insights', 'goal', 'team-onboarding'], 'apiKeySource': 'none', 'claude_code_version': '2.1.150', 'output_style'

In [12]:
async def ask(prompt: str, options) -> str:
    final_text = ""
    async for msg in query(prompt=prompt, options=options):
        if hasattr(msg, "subtype") and msg.subtype == "success":
            final_text = msg.result
    return final_text


In [13]:

final_text = await ask(example["problem"], leaf_options)

print(verify_math_llm_answer(example, final_text))


True
